# 1. Introduction
In this notebook we'll start from a time series of Sentinel-2 data over an agricultural area in Malawi, located in between the towns of Nkhotakota in the east and Kasungu in the west.<br>
We will demonstrate some parcel-based crop monitoring applications, including:

- field delineation
- phenology (growing season) detection

We will further build upon the first exercise in which we pre-processed the timeseries to make it ready for further use.

# 2. Preparations

<div class="alert alert-block alert-warning">
<b>Running this notebook on CDSE notebooks?</b><br>

Make sure you select the **Geo science** python kernel.<br>

Execute the following cell to install some required python packages.

</div>

In [ ]:
!pip install scikit-image numba --quiet

Now we import the necessary functions that will be used in this notebook.

If you want to learn what happens exactly in these functions, please browse to the folder `vito_agri_tutorials` and locate the specific function you're looking for in this folder.<br>
You can find out the location using the relative paths as specified below, for instance:<br>
`interpolate_ts` function is located in vito_agri_tutorials > utils > interpolate.py

In [ ]:
# Import the necessary python libraries...
from pathlib import Path
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from skimage.segmentation import felzenszwalb
from skimage.segmentation import mark_boundaries

# import custom python functions all defined in separate .py files
from vito_agri_tutorials.utils.mask import mask_ts
from vito_agri_tutorials.utils.composite import composite_ts
from vito_agri_tutorials.utils.interpolate import interpolate_ts
from vito_agri_tutorials.utils.features import tsteps
from vito_agri_tutorials.utils.phenology import (detect_seasons, visualize_seasons)

In the next cell, we specify the folder where the data is located we'll be working with in this exercise.<br>
We encourage you to open this `data` folder using the file explorer on the left hand side of your screen and have a look which files are located there!

In [ ]:
# define folder where to find the data for this exercise
indir = Path('./data')

# 3. Data pre-processing

Here we quickly repeat the steps from the previous exercise to prepare the data:

In [ ]:
# open 20 m Sentinel-2 data
infile = str(indir / 'S2_L2A_Malawi_20m_small.nc')
ds = xr.open_dataset(infile)

# We convert it to a data array for further processing...
scl_20 = ds[['SCENECLASSIFICATION']].to_array()

# within the scene classification layer, the following values point
# to clouds/shadows:
scl_mask_values = [1, 3, 8, 9, 10, 11]

# create mask (True = not to be masked, False = to be masked)
mask = np.logical_not(scl_20.isin(scl_mask_values))
mask.attrs = scl_20.attrs.copy()

# Get the 20 m bands from the file
ts_20 = ds[['B05', 'B06', 'B07', 'B11', 'B12']].to_array()

# Apply the mask to the data
ts_20_masked = mask_ts(ts_20, mask)

# apply temporal compositing to the timeseries
ts_20_comp = composite_ts(ts_20_masked, freq=10, window=20, mode='median')

# apply interpolation to the time series
ts_20_fin = interpolate_ts(ts_20_comp)

# Get the 10m bands
infile = str(indir / 'S2_L2A_Malawi_10m_small.nc')
ds = xr.open_dataset(infile)

# Get the 10 m bands from the file
ts_10 = ds[['B02', 'B03', 'B04', 'B08']].to_array()

# Apply the mask to the data
ts_10_masked = mask_ts(ts_10, mask)

# Apply temporal compositing
ts_10_comp = composite_ts(ts_10_masked, freq=10, window=20, mode='median')

# Apply linear interpolation
ts_10_fin = interpolate_ts(ts_10_comp)

# As a final step, we merge all bands together
ts_fin = xr.concat([ts_10_fin, ts_20_fin], dim='variable')
ts_fin

# And we free up some memory before proceeding...
del ts_20, ts_20_fin, ts_20_comp, ts_20_masked
del ts_10, ts_10_fin, ts_10_comp, ts_10_masked

# 4. Computation of NDVI
Now that we have our data ready to go, let's compute a well-known vegetation index used as a basis for many agricultural monitoring applications, i.e. the NDVI.

In [ ]:
# Compute NDVI
# isolate the NIR band (B08)
b08 = ts_fin.sel(variable='B08').values
# isolate the RED band (B04)
b04 = ts_fin.sel(variable='B04').values
# compute the index
ndvi = (b08 - b04) / (b08 + b04)
ndvi.shape

In [ ]:
# Visualize spatially for one particular date
# (we select the 11th available date)
fig, ax = plt.subplots()
ndviplot = plt.imshow(ndvi[10, ...])
fig.colorbar(ndviplot, ax=ax)
plt.show()

In [ ]:
# Visualize temporally for one pixel
# retrieve the time coordinate
time = ts_fin.coords['timestamp'].values

# now plot the time series for our pixel located at position 10,10
fig, ax = plt.subplots()
ax.plot(time, ndvi[:, 10, 10], '-or')
plt.xticks(rotation=45, ha='right')
plt.title('NDVI for pixel (10,10)')
plt.show()

# 5. Field delineation
We can use the temporal profile of NDVI to recognize and delineate individual fields in the landscape (different crops behave differently throughout the season). This is illustrated in the following example.

In [ ]:
# Resample NDVI temporally to six equidistant timesteps
# (this means we summarize the temporal profile from originally 29 values into six values)
ndvi_tsteps = tsteps(ndvi)

# plot these 6 timesteps
fig, ax = plt.subplots(6, 1)
for i in range(6):
    feat_plt = ax[i].imshow(np.squeeze(ndvi_tsteps[i, ...]))
    fig.colorbar(feat_plt, ax=ax[i])
plt.show()

The segmentation algorithm we will be using for this demonstration can only use 3 inputs. So we select visually from the above plots the time steps we believe provide most information to correctly delineate individual fields in the image.

NOTE that other possiblities include to start from all original time steps and condense the information to 3 variables using for instance principal component analysis. This is however beyond the scope of this exercise.

In [ ]:
# use 3 out of 6 timesteps for parcel delineation. Depending on your region of interest
tsteps_selected = [1, 3, 5]
inputs = np.take(ndvi_tsteps, tsteps_selected, axis=0)
inputs = np.moveaxis(inputs, 0, -1)

# apply the Felzenzwalb segmentation algorithm
#NOTE that there are many other segmentation algorithms available in the skimage package
# Felzenszwalb is known for its decent results in combination with
# minimal requirements regarding parameter tuning.
#NOTE that you can change the size of the delineated parcels by altering the
# scale parameter in the algoritm. More information here:
# https://scikit-image.org/docs/stable/api/skimage.segmentation.html#skimage.segmentation.felzenszwalb
segments_fz = felzenszwalb(inputs, scale=100, sigma=0.5)
segments_fz

# the result is a raster containing the segment ID for each pixel.
# let's now visualize these results a bit better...

In [ ]:
# plot result
fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(mark_boundaries(inputs, segments_fz))
ax.set_title("Felzenszwalbs segmentation result")
plt.show()

# 6. Phenology detection

Detection of growing seasons can be done on a pixel-per-pixel basis using the NDVI time series of each individual pixel OR it can be done per parcel using the segments delineated above.
Here we will showcase the latter option.

In [ ]:
# compute the average NDVI profile per parcel
segment_ids = np.unique(segments_fz)
nsegments = len(segment_ids)
ntimes = ndvi.shape[0]
ndvi_segm = np.zeros([ntimes, nsegments], dtype=np.float32)
for i, id in enumerate(segment_ids):
    msk = segments_fz == id
    msk_3d = np.broadcast_to(msk, ndvi.shape)
    ndvi_masked = np.where(msk_3d == 1, ndvi, np.nan)
    ndvi_segm[:, i] = np.nanmean(ndvi_masked, axis=(1, 2))

ndvi_segm.shape
# We now have a 2D array with 28 time steps and 246 segments

In [ ]:
# Now apply phenology detection on this data array
# first make sure the input shape is 3D
# (see documentation of the detect_seasons function)
ndvi_segm = np.expand_dims(ndvi_segm, axis=2)
# get the time coordinates
time = ts_fin.coords['timestamp'].values
# run phenology detection
nseas, sos, mos, eos = detect_seasons(ndvi_segm, time)

# inspect the results of the algorithm:
# nseas contains how many seasons have been identified for each pixel (or segment in our case)
# sos contains the start date of up to 5 detected seasons per pixel
# mos contains the peak date of up to 5 detected seasons per pixel
# eos contains the end date of up to 5 detected seasons per pixel

In [ ]:
# visualize output for one pixel
x, y = 1, 0
visualize_seasons(nseas, sos, mos, eos, x, y,
                    ndvi_segm, time)

# green dot denotes the start, black '+' denotes the peak and a red dot denotes the end of the season.


END OF THE EXERCISE